In [53]:
import pandas as pd
pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.max_rows', 1000)

# Phase 2 – Erweiterte Datenbereinigung und Datenvorbereitung

Die Datensätze wurden bereits in der ersten Projektphase bereinigt.
In diesem Notebook führt unser Team zusätzliche Bereinigungen, Feature Engineering und weitere Anpassungen durch, um die Datenqualität für die anschließende Analyse zu verbessern.

##1.Import der bereinigten Datensätze aus der ersten Projektphase

In [ ]:
# orders.csv
url = "https://drive.google.com/file/d/1Tl2ppeHT3vtRbl981uVokkgUkT7yeej9/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orders = pd.read_csv(path)

# orderlines.csv
url = "https://drive.google.com/file/d/16rK9z_qcwhOabFNKUZGwWVLNtXqh0qsi/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orderlines = pd.read_csv(path)

# products.csv
url = "https://drive.google.com/file/d/1KhKWsg0iWCxjNW5b92bqny1RRy9iy4_D/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
products = pd.read_csv(path)

# branbs.csv
url = "https://drive.google.com/file/d/1LalcVCQZRsEYdgYr0_NyYwa1f2Dm4zPg/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
brands = pd.read_csv(path)

##2.Arbeitskopien erstellen

In [ ]:
# Arbeitskopien der Datensätze erstellen,
# damit die Originaldaten unverändert bleiben.

orders_df =orders.copy()
orderlines_df = orderlines.copy()
products_df = products.copy()
brands_df = brands.copy()

In [ ]:
# Die Spalte "product_id" entfernen,
# da sie ausschließlich den Wert 0 enthält und für die weitere Analyse nicht benötigt wird.

orderlines_df =orderlines_df.drop(columns=['product_id'])

##3.Bereinigung der Zeichenketten (String-Spalten)

 Führende und nachgestellte Leerzeichen, können ggf.  bei der Zusammenführung oder beim Vergleichen zu Fehlern führen.

###3.1. 1- orders

In [ ]:
# Führende und nachgestellte Leerzeichen aus allen Textspalten entfernen.

str_cols = [col for col in orders_df.columns if orders_df[col].dtype == 'object']
for col in str_cols :
  orders_df[col] = orders_df[col].str.strip()

 Spaltennamen vereinheitlichen

In [ ]:
# Spaltenname anpassen, um ihn an die anderen Datensätze anzupassen.
orders_df.rename(columns={'order_id':'id_order'}, inplace=True)

###3.2. 2- orderlines

In [ ]:
# Führende und nachgestellte Leerzeichen aus allen Textspalten entfernen.
str_cols = [col for col in orderlines_df.columns if orderlines_df[col].dtype == 'object']
for col in str_cols :
  orderlines_df[col] = orderlines_df[col].str.strip()

###3.3. 3- products

In [ ]:
# Führende und nachgestellte Leerzeichen aus allen Textspalten entfernen.
str_cols = [col for col in products_df.columns if products_df[col].dtype == 'object']
for col in str_cols :
  products_df[col] = products_df[col].str.strip()

**Ergebnis:**

- wurden entfernt

###3.4. 4- Brands

In [ ]:
# Führende und nachgestellte Leerzeichen aus allen Textspalten entfernen.
str_cols = [col for col in brands_df.columns if brands_df[col].dtype == 'object']
for col in str_cols :
  brands_df[col] = brands_df[col].str.strip()

##4.Products

###4.1. Bereinigung fehlerhafter Preisangaben/Preisformat)

- Im Datensatz wurden fehlerhafte Preisformate gefunden (z. B. `34.56.546`), die nicht direkt in numerische Werte umgewandelt werden konnten.

###4.2. Fehlerhafte Preise identifizieren
Zunächst werden alle Produkte identifiziert, deren Preis mehr als eine Dezimalstelle (einen Punkt) enthält.

In [ ]:
price_mask = products_df['price'].str.count(r'\.') > 1
products_df.loc[price_mask, 'price'].shape[0]

377

**Ergebnis:**

- Anstatt diese Datensätze zu löschen, entschied sich unser Team für eine Korrektur der Preisangaben. Der Grund dafür war, dass diese Produkte mit 2397 Bestellpositionen verknüpft sind, darunter 566 abgeschlossene Bestellungen (Completed).

- Daher wurde die erste fehlerhafte Trennstelle entfernt, die Preise auf zwei Dezimalstellen gerundet und anschließend in numerische Werte umgewandelt.

###4.3. Betroffene Bestellungen analysieren

Anschließend wird geprüft, wie viele Bestellpositionen und Bestellungen von diesen Produkten betroffen sind.

In [ ]:
price_mask = products_df['price'].str.count(r'\.') > 1
sku_list = products_df.loc[price_mask, 'sku']
orderlines_mask = orderlines_df['sku'].isin(sku_list)
umsätze_zwei_punkte=orderlines_df.loc[orderlines_mask, :]
umsätze_zwei_punkte.shape[0]

2367

**Ergebnis:**

- erste fehlerhafte Trennstelle (.)wurde entfernt

###4.4. Auswirkungen auf abgeschlossene Bestellungen

Zur Entscheidungsfindung wird untersucht, wie viele der betroffenen Bestellungen bereits abgeschlossen wurden.

In [ ]:
orders_umsätse_merged = orders_df.merge(umsätze_zwei_punkte, on='id_order',how='inner')
orders_umsätse_merged
orders_umsätse_merged['state'].value_counts()

,count
state,
Shopping Basket,1183
Completed,566
Place Order,359
Pending,205
Cancelled,54


**Ergebnis:**

- 566 Bestellungen haben den Status "Completed"

###4.5. Preisformat korrigieren

Die erste fehlerhafte Trennstelle wird entfernt, sodass die Preise anschließend als numerische Werte gespeichert werden können.

In [ ]:
products_df.loc[price_mask, 'price'] = products_df.loc[price_mask, 'price'].str.replace('.','',1)

**Ergebnis:**

- 1. Trennstelle wurde entfernt

### 4.6. Datentyp umwandeln

Nach der Korrektur wird die Spalte in einen numerischen Datentyp umgewandelt.

In [ ]:
products_df['price'] = pd.to_numeric(products_df['price'])

**Ergebnis:**

- price ist nun float

###4.7. Preise an Euro-Format anpassen

Zum Schluss werden die betroffenen Preise auf zwei Dezimalstellen angepasst.

In [ ]:
price_mask = products_df.price.astype(str).str.contains(r"\d+\.\d{3}")
products_df.loc[price_mask ,'price'] =round(products_df.loc[price_mask , 'price'] / 10, 2)

**Ergebis:**

- price mit 2 Dezimalstellen

###4.8. Spalte-type


In der Spalte **type** wurden 50 fehlende Werte (NaN) gefunden.

Unser Team hat diese Datensätze analysiert und festgestellt, dass ein großer Teil davon Produkte von Apple betrifft.

Da nur wenige Datensätze betroffen sind und kein ausreichender Hinweis für eine korrekte Nachtragung des Produkttyps vorliegt, haben wir uns bewusst gegen das Löschen oder Ersetzen dieser Werte entschieden.

####4.8.1 Fehlende Werte identifizieren

In [ ]:
type_mask = products_df['type'].isna()
type_nan = products_df.loc[type_mask,:]
type_nan.shape[0]

50

**Ergebnis:**

- Die Datensätze bleiben daher unverändert erhalten, um keinen Informationsverlust zu verursachen.

####4.8.2 Auswirkungen auf Bestellungen analysieren

Die Analyse zeigt, dass nur 54 Bestellungen betroffen wären.

Da keine zuverlässige Methode zur Bestimmung des fehlenden Produkttyps vorhanden ist, wurden die Datensätze nicht gelöscht.

Stattdessen wurden die fehlenden Werte in der Spalte **type** mit **„unbekannt“** ersetzt. Dadurch enthält die Tabelle keine NaN-Werte mehr und kann in den weiteren Analysen konsistent verwendet werden.

In [ ]:
orderlines_type_merged = orderlines_df.merge(type_nan, on='sku', how='inner')
#pd.set_option('display.max_rows',None)
orderlines_type_merged.shape[0]

171

In [ ]:
dd = orders_df.merge(orderlines_type_merged, on='id_order', how='inner')
mask = dd['state'].isin(['Completed','Pending','Place Order'])
dd.loc[mask,:].shape[0]

54

In [ ]:
products_df.loc[type_mask,:'type'] = 'unbekannt'

/tmp/ipykernel_1838/1943873664.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'unbekannt' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  products_df.loc[type_mask,:'type'] = 'unbekannt'
/tmp/ipykernel_1838/1943873664.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'unbekannt' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  products_df.loc[type_mask,:'type'] = 'unbekannt'


**Ergebnis:**

- Die fehlenden Werte in der Spalte type mit „unbekannt“ ersetzt.

##5.Erstellung einer zusammengeführten Analysetabelle (.merged)

Für die weitere Analyse wurden die Tabellen **orderlines** und **products** über die Spalte **sku** zusammengeführt.

Anschließend wurde eine neue Variable **discount_percent** berechnet. Sie beschreibt den prozentualen Rabatt eines Produkts zum Zeitpunkt der Bestellung. Grundlage für die Berechnung ist der Vergleich zwischen dem ursprünglichen Produktpreis (**price**) und dem tatsächlichen Verkaufspreis (**unit_price**).

###5.1. orderlines und products zusammenführen (.merged)

In [ ]:
orderlines_products_merged = orderlines_df.merge(products_df, on='sku',how='inner')
orderlines_products_merged.shape[0]

214922

### 5.2.Rabatt pro Bestellung berechnen


In [ ]:
orderlines_products_merged['discount_percent'] = 100 - round((orderlines_products_merged['unit_price'] * 100) /
                                                        orderlines_products_merged['price'], 0)
orderlines_products_merged.head(5)

,id,id_order,product_quantity,sku,unit_price,date,name,desc,price,promo_price,in_stock,type,discount_percent
0,1119109,299539,1,OTT0133,18.99,2017-01-01 00:07:19,Otterbox iPhone Case Symmetry 2.0 SE / 5s / 5 ...,resistant cover and thin beveled edges for iPh...,34.99,199.904,0,11865403,45.73
1,1119110,299540,1,LGE0043,399.00,2017-01-01 00:19:45,"27UD58-B LG Monitor 27 ""4K UHD DisplayPort",Monitor for gamers and multimedia professional...,429.00,3.989.999,0,1296,6.99
2,1119111,299541,1,PAR0071,474.05,2017-01-01 00:20:57,Parrot Bebop 2 White + Command FLYPAD and FPV ...,cuadricóptero wireless remote control with 25 ...,699.00,5.689.989,0,11905404,32.18
3,1119112,299542,1,WDT0315,68.39,2017-01-01 00:51:40,"Blue WD 2TB Hard Drive 35 ""Mac and PC",Internal Hard Drive Western Digital 2TB 3.5-in...,79.00,639.945,0,12655397,13.43
4,1119113,299543,1,JBL0104,23.74,2017-01-01 01:06:38,Gray Bluetooth Speaker JBL GO,Compact Bluetooth Handsfree for iPhone iPad an...,29.90,27.99,1,5398,20.60


Der Rabatt wird mit folgender Formel berechnet:

**Rabatt (%) = 100 − (unit_price × 100 / price)**

Positive Werte bedeuten einen Rabatt gegenüber dem ursprünglichen Produktpreis.
Negative Werte zeigen, dass der Verkaufspreis zum Bestellzeitpunkt höher war als der aktuell gespeicherte Produktpreis.

###5.3.Bestellinformationen ergänzen

Mit orders zusammenführen ergänzen (.merged)

In [ ]:
orders_orderlines_products_merged =orders_df.merge(orderlines_products_merged, on='id_order', how='inner')
orders_orderlines_products_merged.head()

,id_order,created_date,total_paid,state,id,product_quantity,sku,unit_price,date,name,desc,price,promo_price,in_stock,type,discount_percent
0,241319,2017-01-02 13:35:40,44.99,Cancelled,1121139,1,JBL0123,44.99,2017-01-02 12:26:59,JBL T450 Bluetooth BT Headset White,Wireless headphones with folding design with 1...,49.95,429.901,1,5384,9.93
1,241423,2017-11-06 13:10:02,136.15,Completed,1398738,1,LAC0212,129.16,2017-11-06 12:47:20,LaCie Porsche Design Desktop Drive 4TB USB 3.0...,External Hard Drive 4TB 35-inch USB 3.0 for Ma...,139.99,1.149.948,1,11935397,7.74
2,242832,2017-12-31 17:40:03,15.76,Completed,1529178,1,PAR0074,10.77,2017-12-31 17:26:40,Parrot 550mAh battery for MiniDrones,550mAh rechargeable battery for Parrot minidrones,17.99,109.904,0,11905404,40.13
3,243330,2017-02-16 10:59:38,84.98,Completed,1181923,1,OWC0074,77.99,2017-02-15 17:07:44,Mac OWC Memory 8GB 1066MHZ DDR3 SO-DIMM,8GB RAM Mac mini iMac MacBook and MacBook Pro ...,99.99,999.896,1,1364,22.00
4,243784,2017-11-24 13:35:19,157.86,Cancelled,1437153,3,PHI0080,51.29,2017-11-24 13:27:41,Philips HUE White Ambience Pack 2 GU10 bulbs,Pack of two bulbs with different shades of whi...,59.95,589.899,1,11905404,14.45


In [ ]:
orders_orderlines_products_merged = orders_orderlines_products_merged.loc[orders_orderlines_products_merged['state'].isin(['Completed','Pending','Place Order']),:]

###5.4.Verteilung des Bestellstatus

Zum Abschluss wurde die Anzahl der Bestellungen je Status ermittelt.

In [ ]:
bestellungen = orders_orderlines_products_merged['state'].value_counts()
bestellungen

,count
state,
Completed,56402
Place Order,27450
Pending,16963


**Ergebnis:**

- Diese Information dient als Grundlage für die nachfolgenden Analysen der Rabattverteilung.

##6.Export der bereinigten Datensätze

Nach Abschluss aller Bereinigungs- und Vorverarbeitungsschritte werden die bereinigten Datensätze als CSV-Dateien exportiert.

Diese Dateien dienen als Grundlage für die nächste Projektphase (Analyse und Visualisierung).

In [ ]:
#from google.colab import files

orders_df.to_csv("orders_cl_02.csv", index=False)
files.download("orders_cl_02.csv")

orderlines_df.to_csv("orderlines_cl_02.csv", index=False)
files.download("orderlines_cl_02.csv")

products_df.to_csv("products_cl_02.csv", index=False)
files.download("products_cl_02.csv")

brands_df.to_csv("brands_cl.csv_02", index=False)
files.download("brands_cl.csv_02")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>